# 📊 Phase 2 — EDA, Cleaning & Splits
**Project:** Plant Disease Detection System | VIT Bhopal University

**Run this notebook after downloading PlantVillage into `data/raw/PlantVillage/`**

Steps covered:
1. Count images per class → bar chart
2. Visual inspection grid
3. Remove exact & near duplicates
4. Convert all images to RGB JPEG
5. Create stratified 70/15/15 train/val/test splits

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import hashlib
import shutil
import random
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR       = Path('data/raw/PlantVillage')       # input: colour PlantVillage
PROCESSED_DIR = Path('data/processed/PlantVillage') # output: RGB JPEG images
SPLITS_DIR    = Path('data/splits')                  # output: CSV splits
DOCS_DIR      = Path('docs')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('✅ Imports and paths OK')
print(f'   RAW_DIR exists: {RAW_DIR.exists()}')

## Step 2.2 — Count Images Per Class

In [ ]:
# Count images per class
class_counts = {}
all_images = []  # list of (path, label)

for class_dir in sorted(RAW_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    imgs = [f for f in class_dir.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    class_counts[class_dir.name] = len(imgs)
    for img in imgs:
        all_images.append((str(img), class_dir.name))

total = sum(class_counts.values())
print(f'Total classes : {len(class_counts)}')
print(f'Total images  : {total:,}')
print(f'\nSmallest 5 classes:')
for k, v in sorted(class_counts.items(), key=lambda x: x[1])[:5]:
    print(f'  {v:5d}  {k}')

In [ ]:
# Plot horizontal bar chart
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1])
names  = [c[0].replace('___', '\n') for c in sorted_classes]
counts = [c[1] for c in sorted_classes]

fig, ax = plt.subplots(figsize=(12, 14))
bars = ax.barh(names, counts, color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=8)
ax.set_xlabel('Number of Images')
ax.set_title('PlantVillage — Images Per Class (Colour Version)', fontsize=14, fontweight='bold')
ax.axvline(500, color='red', linestyle='--', linewidth=1, label='500 image threshold')
ax.legend()
plt.tight_layout()
out_path = DOCS_DIR / 'class_distribution.png'
plt.savefig(out_path, dpi=120)
plt.show()
print(f'✅ Saved → {out_path}')

## Step 2.3 — Visual Inspection (20 random images per class)

In [ ]:
# Show 20 random images for a chosen class — change CLASS_NAME as needed
CLASS_NAME = list(class_counts.keys())[0]  # change this to inspect different classes

class_dir = RAW_DIR / CLASS_NAME
imgs = [f for f in class_dir.iterdir() if f.suffix.lower() in ('.jpg','.jpeg','.png')]
sample = random.sample(imgs, min(20, len(imgs)))

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle(f'Class: {CLASS_NAME}', fontsize=14, fontweight='bold')
for ax, img_path in zip(axes.flat, sample):
    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Inspected: {CLASS_NAME} ({len(imgs)} images)')

## Step 2.4 — Remove Exact & Near Duplicates

In [ ]:
PHASH_THRESHOLD = 10  # hamming distance — tune if needed

removed = []   # list of dicts for the log CSV
kept_hashes = {}   # md5 → path
kept_phashes = []  # list of (phash, path)

print('Scanning for duplicates...')
for img_path, label in all_images:
    p = Path(img_path)
    # -- Exact duplicate: MD5
    md5 = hashlib.md5(p.read_bytes()).hexdigest()
    if md5 in kept_hashes:
        removed.append({'path': img_path, 'label': label,
                        'reason': 'exact_duplicate', 'hash': md5,
                        'duplicate_of': kept_hashes[md5]})
        continue
    kept_hashes[md5] = img_path

    # -- Near duplicate: pHash
    try:
        ph = imagehash.phash(Image.open(img_path))
    except Exception:
        removed.append({'path': img_path, 'label': label,
                        'reason': 'corrupt_image', 'hash': '', 'duplicate_of': ''})
        continue

    near_dup = next((kph for kph, kp in kept_phashes
                     if abs(ph - kph) <= PHASH_THRESHOLD), None)
    if near_dup is not None:
        removed.append({'path': img_path, 'label': label,
                        'reason': 'near_duplicate',
                        'hash': str(ph), 'duplicate_of': ''})
        continue
    kept_phashes.append((ph, img_path))

print(f'Total removed : {len(removed):,}')
print(f'Total kept    : {len(all_images) - len(removed):,}')

# Save log
log_path = Path('data/removed_duplicates.csv')
log_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(removed).to_csv(log_path, index=False)
print(f'✅ Log saved → {log_path}')

## Step 2.5 — Convert to RGB JPEG & Step 2.6 — Create Splits

In [ ]:
removed_paths = {r['path'] for r in removed}
clean_images = [(p, l) for p, l in all_images if p not in removed_paths]

processed_records = []  # (new_path, label)

print(f'Converting {len(clean_images):,} images to RGB JPEG...')
for img_path, label in clean_images:
    p = Path(img_path)
    out_dir = PROCESSED_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (p.stem + '.jpg')
    if not out_path.exists():
        try:
            img = Image.open(img_path).convert('RGB')
            img.save(out_path, 'JPEG', quality=95)
        except Exception as e:
            print(f'  ⚠️  Skipped {img_path}: {e}')
            continue
    processed_records.append((str(out_path), label))

print(f'✅ Conversion done. {len(processed_records):,} images in {PROCESSED_DIR}')

In [ ]:
# Stratified 70/15/15 split
df = pd.DataFrame(processed_records, columns=['path', 'label'])

train_rows, temp_rows = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_rows, test_rows = train_test_split(
    temp_rows, test_size=0.50, stratify=temp_rows['label'], random_state=SEED)

train_rows.to_csv(SPLITS_DIR / 'train.csv', index=False)
val_rows.to_csv(SPLITS_DIR / 'val.csv',   index=False)
test_rows.to_csv(SPLITS_DIR / 'test.csv',  index=False)

print(f'✅ Splits saved to {SPLITS_DIR}/')
print(f'   Train : {len(train_rows):,}')
print(f'   Val   : {len(val_rows):,}')
print(f'   Test  : {len(test_rows):,}')
print(f'\n⚠️  NEVER reshuffle these CSVs — seed is fixed at {SEED}')

## ✅ Done!
Now complete Steps 2.7 (field test set from PlantDoc) and 2.8 (fill data_card.md) manually.

**Next:** Run Phase 3 — Model Training (`notebooks/02_train_baseline.ipynb`)